# 02 Planning documents

Transcribes the figures the RMR Master Plan Update (2019, S034), its Appendix A (S011), and the City's 2024 Housing Needs Report (S013) actually state, into structured tables other steps can read instead of re-parsing the source PDFs. This notebook does not calculate anything of its own; every value here traces to a claim ID, and step 03/05/06 do the modeling that uses these tables.

In [ ]:
research_dir = "research"
output_dir = "data/processed"

## Load the ledger

Same guarded loader as every other notebook, so a CSV quoting error raises here rather than silently under-counting claims.

In [ ]:
import sys

sys.path.insert(0, "src")
from resort.ledger import read_ledger

ledger = read_ledger(
    claims_path=f"{research_dir}/claims.csv",
    sources_path=f"{research_dir}/sources.csv",
)
claims = ledger["claims"]
print(f"{len(claims)} claims loaded")

## CCC by phase (from Appendix A, via claim C025)

The appendix's own tables give one total per phase. Written as a small table rather than four separate variables so step 03 can join it against whatever it derives from the per-lift data.

In [ ]:
import pandas as pd

assert claims["C025"]["source_id"] == "S011"
ccc_by_phase = pd.DataFrame(
    [
        {"phase": "Phase 1 (Existing)", "ccc_skiers": 4545},
        {"phase": "Phase 2", "ccc_skiers": 9529},
        {"phase": "Phase 3", "ccc_skiers": 11572},
        {"phase": "Buildout", "ccc_skiers": 18167},
    ]
)
ccc_by_phase

## Employee housing, Phase 2 plan (claim C031)

The master plan states a range of beds per building, not a single number, and a minimum building count, not a maximum. Both bounds are kept in the code cell below rather than picking a midpoint, since picking one would be a judgment call this notebook should not make silently.

In [ ]:
assert claims["C031"]["source_id"] == "S034"
employee_housing_phase2 = {
    "min_buildings": 3,
    "beds_per_building_min": 150,
    "beds_per_building_max": 200,
    "lower_village_hectares": 9.19,
    "total_beds_min": 3 * 150,
    "total_beds_max": 3 * 200,
}
employee_housing_phase2

## Housing need, 5-year and 20-year (claims C026-C029)

Figure 26 of the 2024 HNR gives six components for both windows; kept as a full table, not just the two totals, since step 06 needs the components (the vacancy-rate component in particular ties to C029's separate rental-vacancy finding).

In [ ]:
for claim_id in ("C026", "C027", "C028", "C029"):
    assert claims[claim_id]["source_id"] == "S013"

housing_need = pd.DataFrame(
    [
        {"component": "A: Extreme core housing need", "years_5": 19, "years_20": 75},
        {"component": "B: Homelessness", "years_5": 13, "years_20": 26},
        {"component": "C: Suppressed household formation", "years_5": 86, "years_20": 345},
        {"component": "D: Household growth", "years_5": 504, "years_20": 1155},
        {"component": "E: Vacancy rate adjustment", "years_5": 4, "years_20": 14},
        {"component": "F: Demand buffer", "years_5": 188, "years_20": 752},
    ]
)
housing_need

## Write outputs

One file per table, so step 03 (CCC), step 05 (employee housing), and step 06 (housing need) each read only what they need.

In [ ]:
import json
import os

os.makedirs(output_dir, exist_ok=True)
ccc_by_phase.to_csv(f"{output_dir}/02_ccc_by_phase.csv", index=False, encoding="utf-8")
housing_need.to_csv(f"{output_dir}/02_housing_need_by_component.csv", index=False, encoding="utf-8")
with open(f"{output_dir}/02_employee_housing_phase2.json", "w", encoding="utf-8") as f:
    json.dump(employee_housing_phase2, f, indent=2)
print("wrote 02_ccc_by_phase.csv, 02_housing_need_by_component.csv, 02_employee_housing_phase2.json")

## Checks

The component columns must sum to the report's own stated totals for each window; if they do not, either a transcription error was made here or the source's own total is inconsistent with its components, and either way this should fail loudly rather than pass a wrong total downstream.

In [ ]:
assert housing_need["years_5"].sum() == 814, housing_need["years_5"].sum()
assert housing_need["years_20"].sum() == 2367, housing_need["years_20"].sum()
assert ccc_by_phase["ccc_skiers"].is_monotonic_increasing, "CCC should rise phase over phase"
print("checks passed")

## Versions

In [ ]:
import importlib.metadata
import sys

print("python", sys.version)
for pkg in ["pandas"]:
    print(pkg, importlib.metadata.version(pkg))